# 12 · So sánh điểm mix học với luật DJ (ticket #24)

Câu hỏi: dữ liệu có "học lại" được luật DJ không? Bốn phép đo, đều trên MPD:

1. **Tương quan Spearman** giữa điểm mix học (#23) và điểm mix theo luật (#12) trên các cặp bài ngẫu nhiên, kèm tương quan của từng điểm với ΔBPM, khoảng cách Camelot, |Δenergy|.
2. **Heatmap** điểm trung bình theo khoảng cách Camelot × dải ΔBPM, cho cả điểm học và luật.
3. **Cặp liền nhau thật so với ngẫu nhiên**: tỉ lệ cặp mix được của playlist val so với cùng các bài đang phát ghép với bài kế tiếp ngẫu nhiên.
4. **Recall@10 trong batch** trên cặp val, cho điểm học, luật và mức ngẫu nhiên (cùng định nghĩa với lúc huấn luyện #23).

Mọi phép tính nằm trong `src/rerank/mix_compare.py` (có test); notebook chỉ gọi hàm và vẽ hình.

**Input** (Add Input): `titeohihi/mixnext-splited` (tracks, sessions) và Dataset chứa `mix_learned.npz` của notebook 08 (`mixnext-mix-learned`).

**Output** (`/kaggle/working/out/learned_vs_rule/`): `heatmap.png`, `real_vs_random.png`, `recall.png`, các bảng `.csv` và `summary.json`.

Bật **Internet** để clone repo. Chỉ cần CPU, khoảng 5–10 phút.

In [ ]:
# Bootstrap: lấy code từ GitHub (#24 đã merge vào main).
BRANCH = "main"
REPO = "/kaggle/working/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
!pip install -q --no-deps -e {REPO}
import sys; sys.path.insert(0, REPO)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.eval.run import VAL
from src.rerank.mix_compare import heatmap_table, in_batch_recall, real_vs_random, sample_pairs, score_pairs
from src.rerank.mix_learned import LearnedMix, consecutive_pairs
from src.rerank.mix_rule import RuleMix

DATA = sorted(Path("/kaggle/input").rglob("sessions.parquet"))[0].parent
WEIGHTS = sorted(Path("/kaggle/input").rglob("mix_learned.npz"))[0]
OUT = Path("/kaggle/working/out/learned_vs_rule")
OUT.mkdir(parents=True, exist_ok=True)
print("input:", DATA, "|", WEIGHTS)

tracks = pd.read_parquet(DATA / "tracks.parquet")
sessions = pd.read_parquet(DATA / "sessions.parquet")
scorers = {"learned": LearnedMix.load(tracks, WEIGHTS), "rule": RuleMix(tracks)}
summary = {}

In [ ]:
# 1. Spearman trên cặp ngẫu nhiên: 2.000 bài đang phát × 100 ứng viên
rng = np.random.default_rng(0)
scored = score_pairs(sample_pairs(tracks["track_id"], 2_000, 100, rng), tracks, scorers)
corr = scored[["learned", "rule", "bpm_delta", "camelot_distance", "energy_gap"]].corr(method="spearman")
print(corr.loc[["learned", "rule"]].round(3).to_string())
print()
print(scored[["learned", "rule"]].describe(percentiles=[0.1, 0.5, 0.9]).round(3).to_string())
summary["spearman"] = {"n_pairs": len(scored), **corr.loc[["learned", "rule"]].round(4).to_dict(orient="index")}
corr.to_csv(OUT / "spearman.csv")

In [ ]:
# 2. Heatmap: điểm trung bình theo khoảng cách Camelot (0–7) × dải ΔBPM. Mỗi bảng một thang màu riêng:
#    điểm học co cụm (0,5–0,8) nên dùng chung thang 0–1 với luật thì hình học sẽ phẳng một màu.
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

BLUE = LinearSegmentedColormap.from_list("blue", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#0d366b"])
BLUE.set_bad("#f0efec")  # ô không có cặp nào
INK, MUTED = "#0b0b0b", "#52514e"

tables = {name: heatmap_table(scored, name) for name in scorers}
counts = heatmap_table(scored, "learned", aggfunc="count").fillna(0).astype(int)
titles = {"learned": "Điểm mix học (#23)", "rule": "Điểm mix theo luật (#12)"}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, (name, table) in zip(axes, tables.items()):
    im = ax.imshow(table.to_numpy(dtype=float), cmap=BLUE, aspect="auto")
    lo, hi = np.nanmin(table.to_numpy(dtype=float)), np.nanmax(table.to_numpy(dtype=float))
    for (i, j), v in np.ndenumerate(table.to_numpy(dtype=float)):
        if np.isfinite(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="white" if v > lo + 0.6 * (hi - lo) else INK)
    ax.set_xticks(range(table.shape[1]), table.columns)
    ax.set_yticks(range(table.shape[0]), table.index)
    ax.set_xlabel("ΔBPM (xét cả ×2/×0,5)", color=MUTED)
    ax.set_ylabel("Khoảng cách Camelot (bước vòng + đổi A/B)", color=MUTED)
    ax.set_title(titles[name], color=INK, loc="left")
    ax.tick_params(colors=MUTED, length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    fig.colorbar(im, ax=ax, shrink=0.8, label="điểm trung bình")
fig.savefig(OUT / "heatmap.png", dpi=200)
plt.show()
for name, table in tables.items():
    table.to_csv(OUT / f"heatmap_{name}.csv")
counts.to_csv(OUT / "heatmap_counts.csv")
print("Số cặp mỗi ô:\n", counts.to_string())

In [ ]:
# 3. Playlist thật có mượt hơn ngẫu nhiên không? Cặp liền nhau trong phiên val so với bài kế tiếp ngẫu nhiên
smooth = real_vs_random(sessions, tracks, split=VAL, rng=np.random.default_rng(0), repeats=1)
print(smooth.round(3).to_string())
smooth.to_csv(OUT / "real_vs_random.csv")
summary["real_vs_random"] = smooth.round(4).to_dict(orient="index")

metrics = {"mixable_pct": "Cặp mix được (%)", "key_compatible_pct": "Tông tương thích (%)"}
fig, ax = plt.subplots(figsize=(7, 3.6), constrained_layout=True)
x = np.arange(len(metrics))
for k, (row, color) in enumerate([("thật", "#2a78d6"), ("ngẫu nhiên", "#eb6834")]):
    vals = [smooth.loc[row, m] for m in metrics]
    bars = ax.bar(x + (k - 0.5) * 0.36, vals, width=0.34, color=color, label=f"cặp {row}", edgecolor="white", linewidth=2)
    ax.bar_label(bars, fmt="%.1f", padding=2, color=INK, fontsize=9)
ax.set_xticks(x, list(metrics.values()))
ax.set_ylabel("%", color=MUTED)
ax.set_title("Cặp liền nhau trong playlist val so với bài kế tiếp ngẫu nhiên", color=INK, loc="left")
ax.legend(frameon=False)
ax.grid(axis="y", color="#e6e5e0")
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
fig.savefig(OUT / "real_vs_random.png", dpi=200)
plt.show()

In [ ]:
# 4. Recall@10 trong batch trên cặp val (trộn như lúc huấn luyện, batch 1.024): điểm học, luật, ngẫu nhiên.
#    Mẫu 20.480 cặp val ở đây khác mẫu 50.000 cặp lúc huấn luyện, nên Recall@10 của điểm học sẽ gần, nhưng không
#    trùng đúng, con số 0,0351 trong mix_learned.json. Điểm học và luật được so trên cùng các batch.
val_pairs = consecutive_pairs(sessions, VAL)
val_pairs = val_pairs.iloc[np.random.default_rng(0).permutation(len(val_pairs))[:20_480]]
recall = {name: in_batch_recall(scorer, val_pairs, batch_size=1024) for name, scorer in scorers.items()}
table = pd.DataFrame(recall).T
print(table.round(4).to_string())
summary["in_batch_recall_at_10"] = table.round(5).to_dict(orient="index")

labels = {"learned": "Điểm mix học", "rule": "Điểm mix theo luật"}
names = [labels[n] for n in recall] + ["Ngẫu nhiên"]
values = [r["recall"] for r in recall.values()] + [table["random_recall"].mean()]
fig, ax = plt.subplots(figsize=(7, 3.2), constrained_layout=True)
bars = ax.barh(names, values, color=["#2a78d6"] * len(recall) + ["#b5b3ad"], height=0.55)
ax.bar_label(bars, fmt="%.4f", padding=3, color=INK, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Recall@10 của bài kế tiếp thật trong batch 1.024 cặp val", color=MUTED)
ax.set_title("Hàm mix nào đoán đúng bài kế tiếp hơn?", color=INK, loc="left")
ax.set_xlim(0, max(values) * 1.18)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
fig.savefig(OUT / "recall.png", dpi=200)
plt.show()

In [ ]:
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
for p in sorted(OUT.iterdir()):
    print(p.name, f"{p.stat().st_size / 1e3:.1f} KB")

## Đăng output

**Save Version → Save & Run All (Commit)**. Ba hình trong `out/learned_vs_rule/` dùng cho báo cáo (#29); không cần đăng Dataset.

## Kết quả (Kaggle, CPU, 27/11/2026)

**1. Spearman trên 200.000 cặp ngẫu nhiên** (2.000 bài đang phát × 100 ứng viên)

| | với điểm luật | với ΔBPM | với khoảng cách Camelot | với \|Δenergy\| |
|---|---:|---:|---:|---:|
| Điểm mix học | **0,153** | −0,036 | −0,076 | **−0,288** |
| Điểm mix theo luật | 1 | −0,600 | −0,366 | −0,375 |

**2. Heatmap** (`heatmap.png`): điểm học giảm đều theo khoảng cách Camelot (khoảng 0,67 ở 0–1 bước xuống 0,63 ở 7 bước), gần như phẳng theo ΔBPM (chênh ≤ 0,01 giữa "≤ 3%" và "> 20%"). Điểm luật giảm mạnh theo cả hai trục (0,95 → 0,15). Ô ít nhất có 625 cặp.

**3. Cặp liền nhau trong playlist val so với bài kế tiếp ngẫu nhiên** (562.517 cặp)

| | cặp thật | cặp ngẫu nhiên | chênh |
|---|---:|---:|---:|
| Cặp mix được (%) | **4,78** | 3,39 | ×1,41 |
| Tông tương thích (%) | 21,67 | 19,09 | +2,6 điểm % |
| ΔBPM trung bình (%) | 15,74 | 16,97 | −1,2 điểm % |
| \|Δenergy\| trung bình | **0,171** | 0,227 | −25% |

**4. Recall@10 trong batch** (20.480 cặp val, batch 1.024): điểm học **0,0364**, điểm luật **0,0196**, ngẫu nhiên 0,0098. Điểm học khớp lúc huấn luyện (0,0351 trên mẫu khác).

## Kết luận cho mục Tranh luận

Dữ liệu **học lại được một phần** luật DJ, và phần học lại được khớp với cách người tạo playlist thật sự nối bài:

- **Playlist thật có mượt hơn ngẫu nhiên, nhưng ít.** Tỉ lệ cặp mix được 4,78% so với 3,39% (gấp 1,4 lần). Khác biệt rõ nhất là năng lượng (|Δenergy| thấp hơn 25%), sau đó là tông (+2,6 điểm %); BPM gần như không khác (−1,2 điểm %).
- **Luật DJ có mang tín hiệu về bài kế tiếp thật:** Recall@10 gấp 2 lần ngẫu nhiên. **Điểm mix học mạnh hơn** (gấp 3,7 lần) vì học trực tiếp từ hành vi.
- **Điểm học cùng chiều với luật ở tông và năng lượng, nhưng gần như bỏ qua BPM.** Nó phụ thuộc chủ yếu vào |Δenergy| (Spearman −0,29), ít vào khoảng cách Camelot (−0,08), gần như không vào ΔBPM (−0,04); trong khi luật đặt nặng BPM nhất (−0,60, w_b = 0,4). Đúng với mục 3: người tạo playlist giữ năng lượng liền mạch, khớp tông vừa phải, còn khớp nhịp là thói quen riêng của DJ. Vì vậy tương quan tổng giữa hai điểm thấp (0,153).
- **Hệ quả cho bước xếp lại:** muốn tăng tỉ lệ cặp mix được (mục tiêu của đề tài) thì phải dùng luật, vì điểm học không đẩy về phía khớp BPM. Điểm học hợp hơn nếu mục tiêu là "nối giống người thật". Điểm học co cụm (độ lệch chuẩn 0,087 so với 0,215 của luật), nên cùng một α nó tác động lên thứ hạng yếu hơn (#25).

**Giới hạn** (spec §6): cặp bài liền nhau trong playlist phản ánh cả gu lẫn độ chuyển bài, nên điểm mix học không thuần là "độ mượt". Hai bài đứng cạnh nhau có thể chỉ vì cùng nghệ sĩ, cùng thể loại hay cùng thời kỳ; đầu vào chỉ có BPM, tông và energy nên điểm học không phân biệt được hai lý do này. Tín hiệu cũng yếu: loss InfoNCE chỉ thấp hơn mức chấm đều 0,16 nat (#23). Kết quả trên một bộ dữ liệu (MPD, playlist do người dùng tự sắp); #57 kiểm tra lại trên lịch sử nghe last.fm.